In [0]:
### consider data as watermark column
### when data matched don`t push to destination 
### when data is not matching then write a new row in the destination table

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp

spark = SparkSession.builder.getOrCreate()

source_data = [
    (101, "Ravi", 5000, "2026-10-20 10:00:00"),
    (102, "Priya", 7500, "2026-10-25 11:00:00"),
    (103, "Arjun", 3000, "2026-10-20 12:00:00"),
    (104, "Sneha", 9000, "2026-10-25 13:00:00"),
    (105, "Kiran", 4500, "2026-10-25 14:00:00"),
    (106, "Meena", 6200, "2026-10-25 15:00:00")
]

columns = [
    "order_id", "customer", "amount", "updated_at"
]

source_df = spark.createDataFrame(source_data, columns)

source_df = source_df.withColumn(
    "updated_at", to_timestamp("updated_at")
)

source_df.show(truncate=False)

+--------+--------+------+-------------------+
|order_id|customer|amount|updated_at         |
+--------+--------+------+-------------------+
|101     |Ravi    |5000  |2026-10-20 10:00:00|
|102     |Priya   |7500  |2026-10-25 11:00:00|
|103     |Arjun   |3000  |2026-10-20 12:00:00|
|104     |Sneha   |9000  |2026-10-25 13:00:00|
|105     |Kiran   |4500  |2026-10-25 14:00:00|
|106     |Meena   |6200  |2026-10-25 15:00:00|
+--------+--------+------+-------------------+



In [0]:
from pyspark.sql import SparkSession

# ==========================================
# STEP 1: CREATE SPARK SESSION
# ==========================================

spark = SparkSession.builder \
    .appName("IncrementalLoading") \
    .getOrCreate()


# ==========================================
# STEP 2: CREATE SOURCE TABLE (5 RECORDS)
# ==========================================

columns = ["id", "name", "salary"]

source_data = [
    (1, "Ravi", 5000),
    (2, "Priya", 6000),
    (3, "Arjun", 7000),
    (4, "Sneha", 8000),
    (5, "Kiran", 9000)
]

source_df = spark.createDataFrame(source_data, columns)

print("STEP 2: INITIAL SOURCE TABLE")
source_df.orderBy("id").show()


# ==========================================
# STEP 3: CREATE TARGET TABLE (3 RECORDS)
# ==========================================

target_data = [
    (1, "Ravi", 5000),
    (2, "Priya", 6000),
    (3, "Arjun", 7000)
]

target_df = spark.createDataFrame(target_data, columns)

print("STEP 3: INITIAL TARGET TABLE")
target_df.orderBy("id").show()


# ==========================================
# STEP 4: FIRST INCREMENTAL LOAD
# ==========================================

# Find records in source but not in target
incremental_df = source_df.join(
    target_df,
    on="id",
    how="left_anti"
)

print("STEP 4: FIRST INCREMENTAL RECORDS")
incremental_df.orderBy("id").show()

# Insert incremental records into target
target_df = target_df.unionByName(incremental_df)

print("TARGET AFTER FIRST INCREMENTAL LOAD")
target_df.orderBy("id").show()


# ==========================================
# STEP 5: INSERT 3 NEW RECORDS INTO SOURCE
# ==========================================

new_data = [
    (6, "Meena", 10000),
    (7, "Rahul", 11000),
    (8, "Divya", 12000)
]

new_df = spark.createDataFrame(new_data, columns)

# Append new records to source
source_df = source_df.unionByName(new_df)

print("STEP 5: SOURCE AFTER INSERTING NEW ROWS")
source_df.orderBy("id").show()


# ==========================================
# STEP 6: SECOND INCREMENTAL LOAD
# ==========================================

incremental_df = source_df.join(
    target_df,
    on="id",
    how="left_anti"
)

print("STEP 6: SECOND INCREMENTAL RECORDS")
incremental_df.orderBy("id").show()

# Insert only new records
target_df = target_df.unionByName(incremental_df)

print("TARGET AFTER SECOND INCREMENTAL LOAD")
target_df.orderBy("id").show()


# ==========================================
# STEP 7: INSERT 2 MORE RECORDS INTO SOURCE
# ==========================================

third_batch_data = [
    (9, "Vijay", 13000),
    (10, "Anjali", 14000)
]

third_batch_df = spark.createDataFrame(
    third_batch_data,
    columns
)

source_df = source_df.unionByName(third_batch_df)

print("STEP 7: SOURCE AFTER THIRD INSERT")
source_df.orderBy("id").show()


# ==========================================
# STEP 8: THIRD INCREMENTAL LOAD
# ==========================================

incremental_df = source_df.join(
    target_df,
    on="id",
    how="left_anti"
)

print("STEP 8: THIRD INCREMENTAL RECORDS")
incremental_df.orderBy("id").show()

target_df = target_df.unionByName(incremental_df)


# ==========================================
# STEP 9: DISPLAY FINAL SOURCE AND TARGET
# ==========================================

print("FINAL SOURCE TABLE")
source_df.orderBy("id").show()

print("FINAL TARGET TABLE")
target_df.orderBy("id").show()


# ==========================================
# STEP 10: VERIFY RECORD COUNTS
# ==========================================

print("SOURCE COUNT:", source_df.count())
print("TARGET COUNT:", target_df.count())


# ==========================================
# STEP 11: CHECK IF ANY NEW RECORDS EXIST
# ==========================================

remaining_df = source_df.join(
    target_df,
    on="id",
    how="left_anti"
)

print("REMAINING INCREMENTAL RECORDS")
remaining_df.show()

print("REMAINING RECORD COUNT:", remaining_df.count())

# Expected output: 0
# All new records have been loaded successfully.

In [0]:

# ============================================================
# INCREMENTAL LOADING USING PYSPARK AND DELTA LAKE
# WHEN MATCHED -> UPDATE
# WHEN NOT MATCHED -> INSERT
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ------------------------------------------------------------
# STEP 1: CREATE SOURCE TABLE WITH DUMMY DATA
# ------------------------------------------------------------

source_data = [
    (101, "Rahul", 1000, "Pending",   "2026-10-20 10:00:00"),
    (102, "Priya", 2000, "Pending",   "2026-10-20 11:00:00"),
    (103, "Amit",  3000, "Shipped",   "2026-10-20 12:00:00"),
    (104, "Sneha", 4000, "Delivered", "2026-10-20 13:00:00"),
    (105, "Kiran", 5000, "Pending",   "2026-10-20 14:00:00")
]

columns = [
    "order_id",
    "customer_name",
    "amount",
    "status",
    "updated_at"
]

source_df = spark.createDataFrame(source_data, columns)

source_df = source_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)

# Save source Delta table
source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("source_orders")

print("INITIAL SOURCE TABLE")
spark.table("source_orders").orderBy("order_id").show()


# ------------------------------------------------------------
# STEP 2: INITIAL FULL LOAD INTO TARGET
# ------------------------------------------------------------

spark.table("source_orders") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("target_orders")

print("INITIAL TARGET TABLE")
spark.table("target_orders").orderBy("order_id").show()


# ------------------------------------------------------------
# STEP 3: GET LAST PROCESSED TIMESTAMP (WATERMARK)
# ------------------------------------------------------------

last_watermark = spark.table("target_orders") \
    .agg(F.max("updated_at").alias("last_watermark")) \
    .x()["last_watermark"]

print("LAST WATERMARK:", last_watermark)


# ------------------------------------------------------------
# STEP 4: INSERT NEW RECORDS INTO SOURCE
# ------------------------------------------------------------

new_data = [
    (106, "Vijay",  6000, "Pending", "2026-10-21 09:00:00"),
    (107, "Anjali", 7000, "Pending", "2026-10-21 10:00:00")
]

new_df = spark.createDataFrame(new_data, columns)

new_df = new_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)

new_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("source_orders")


# ------------------------------------------------------------
# STEP 5: UPDATE EXISTING RECORDS IN SOURCE
# ------------------------------------------------------------

# Order 102: Pending -> Shipped, amount 2000 -> 2500
spark.sql("""
    UPDATE source_orders
    SET amount = 2500,
        status = 'Shipped',
        updated_at = TIMESTAMP '2026-10-21 11:00:00'
    WHERE order_id = 102
""")

# Order 105: Pending -> Delivered
spark.sql("""
    UPDATE source_orders
    SET status = 'Delivered',
        updated_at = TIMESTAMP '2026-10-21 12:00:00'
    WHERE order_id = 105
""")

print("SOURCE AFTER INSERT AND UPDATE")
spark.table("source_orders").orderBy("order_id").show()


# ------------------------------------------------------------
# STEP 6: READ ONLY INCREMENTAL RECORDS
# ------------------------------------------------------------

incremental_df = spark.table("source_orders").filter(
    F.col("updated_at") > F.lit(last_watermark)
)

print("INCREMENTAL RECORDS")
incremental_df.orderBy("order_id").show()

# Expected records:
# 102 -> Updated
# 105 -> Updated
# 106 -> New
# 107 -> New


# ------------------------------------------------------------
# STEP 7: MERGE USING WHEN MATCHED AND WHEN NOT MATCHED
# ------------------------------------------------------------

target = DeltaTable.forName(spark, "target_orders")

(
    target.alias("t")
    .merge(
        incremental_df.alias("s"),
        "t.order_id = s.order_id"
    )

    # WHEN MATCHED: UPDATE EXISTING RECORDS
    .whenMatchedUpdate(
        condition="s.updated_at > t.updated_at",
        set={
            "customer_name": "s.customer_name",
            "amount": "s.amount",
            "status": "s.status",
            "updated_at": "s.updated_at"
        }
    )

    # WHEN NOT MATCHED: INSERT NEW RECORDS
    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "customer_name": "s.customer_name",
            "amount": "s.amount",
            "status": "s.status",
            "updated_at": "s.updated_at"
        }
    )

    .execute()
)


# ------------------------------------------------------------
# STEP 8: DISPLAY FINAL TARGET
# ------------------------------------------------------------

print("FINAL TARGET AFTER INCREMENTAL LOAD")

spark.table("target_orders") \
    .orderBy("order_id") \
    .show(truncate=False)


# ------------------------------------------------------------
# STEP 9: CHECK TARGET RECORD COUNT
# ------------------------------------------------------------

print(
    "TOTAL TARGET RECORDS:",
    spark.table("target_orders").count()
)

# Expected: 7


# ------------------------------------------------------------
# STEP 10: VERIFY DUPLICATE RECORDS
# ------------------------------------------------------------

spark.table("target_orders") \
    .groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


# ------------------------------------------------------------
# STEP 11: UPDATE WATERMARK AFTER SUCCESSFUL MERGE
# ------------------------------------------------------------

new_watermark = incremental_df.agg(
    F.max("updated_at").alias("max_timestamp")
).first()["max_timestamp"]

if new_watermark is not None:
    last_watermark = new_watermark

print("NEW WATERMARK:", last_watermark)


# ------------------------------------------------------------
# STEP 12: CHECK NEXT INCREMENTAL LOAD
# ------------------------------------------------------------

next_incremental_df = spark.table("source_orders").filter(
    F.col("updated_at") > F.lit(last_watermark)
)

print("NEXT INCREMENTAL LOAD RECORDS")
next_incremental_df.show()

# Expected: 0 records

# ============================================================
# END
# ============================================================


In [0]:

# ============================================================
# INCREMENTAL LOADING WITH AUTO-GENERATED UNIQUE ID
# PYSPARK + DELTA LAKE
# WHEN MATCHED -> UPDATE
# WHEN NOT MATCHED -> INSERT WITH NEW UUID
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ------------------------------------------------------------
# STEP 1: CREATE DUMMY SOURCE TABLE
# ------------------------------------------------------------

source_data = [
    ("Rahul",  1000, "Pending",   "2026-10-20 10:00:00"),
    ("Priya",  2000, "Pending",   "2026-10-20 11:00:00"),
    ("Amit",   3000, "Shipped",   "2026-10-20 12:00:00"),
    ("Sneha",  4000, "Delivered", "2026-10-20 13:00:00"),
    ("Kiran",  5000, "Pending",   "2026-10-20 14:00:00")
]

columns = [
    "customer_name",
    "amount",
    "status",
    "updated_at"
]

source_df = spark.createDataFrame(source_data, columns)

source_df = source_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)

source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("source_customers")

print("INITIAL SOURCE")
spark.table("source_customers").show(truncate=False)


# ------------------------------------------------------------
# STEP 2: GENERATE UNIQUE ID FOR INITIAL TARGET
# ------------------------------------------------------------

# UUID generates a unique string for each record.
# The ID is generated only when inserting the record.

target_df = spark.table("source_customers") \
    .withColumn("record_id", F.expr("uuid()")) \
    .select(
        "record_id",
        "customer_name",
        "amount",
        "status",
        "updated_at"
    )

target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("target_customers")

print("INITIAL TARGET WITH GENERATED UNIQUE IDS")
spark.table("target_customers").show(truncate=False)


# ------------------------------------------------------------
# STEP 3: GET LAST WATERMARK
# ------------------------------------------------------------

last_watermark = spark.table("target_customers") \
    .agg(F.max("updated_at").alias("watermark")) \
    .first()["watermark"]

print("LAST WATERMARK:", last_watermark)


# ------------------------------------------------------------
# STEP 4: INSERT NEW CUSTOMERS INTO SOURCE
# ------------------------------------------------------------

new_data = [
    ("Vijay",  6000, "Pending", "2026-10-21 09:00:00"),
    ("Anjali", 7000, "Pending", "2026-10-21 10:00:00")
]

new_df = spark.createDataFrame(new_data, columns)

new_df = new_df.withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)

new_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("source_customers")


# ------------------------------------------------------------
# STEP 5: UPDATE EXISTING CUSTOMERS IN SOURCE
# ------------------------------------------------------------

spark.sql("""
    UPDATE source_customers
    SET amount = 2500,
        status = 'Shipped',
        updated_at = TIMESTAMP '2026-10-21 11:00:00'
    WHERE customer_name = 'Priya'
""")

spark.sql("""
    UPDATE source_customers
    SET status = 'Delivered',
        updated_at = TIMESTAMP '2026-10-21 12:00:00'
    WHERE customer_name = 'Kiran'
""")

print("SOURCE AFTER INSERTS AND UPDATES")
spark.table("source_customers").show(truncate=False)


# ------------------------------------------------------------
# STEP 6: GET INCREMENTAL RECORDS
# ------------------------------------------------------------

incremental_df = spark.table("source_customers").filter(
    F.col("updated_at") > F.lit(last_watermark)
)

print("INCREMENTAL RECORDS")
incremental_df.show(truncate=False)


# ------------------------------------------------------------
# STEP 7: MERGE USING UNIQUE CUSTOMER NAME
# ------------------------------------------------------------

target = DeltaTable.forName(
    spark,
    "target_customers"
)

(
    target.alias("t")
    .merge(
        incremental_df.alias("s"),
        "t.customer_name = s.customer_name"
    )

    # MATCHED: UPDATE, PRESERVE EXISTING RECORD ID
    .whenMatchedUpdate(
        condition="s.updated_at > t.updated_at",
        set={
            "amount": "s.amount",
            "status": "s.status",
            "updated_at": "s.updated_at"
        }
    )

    # NOT MATCHED: INSERT WITH AUTO-GENERATED UNIQUE ID
    .whenNotMatchedInsert(
        values={
            "record_id": "uuid()",
            "customer_name": "s.customer_name",
            "amount": "s.amount",
            "status": "s.status",
            "updated_at": "s.updated_at"
        }
    )

    .execute()
)


# ------------------------------------------------------------
# STEP 8: DISPLAY FINAL TARGET
# ------------------------------------------------------------

print("FINAL TARGET")

spark.table("target_customers") \
    .orderBy("customer_name") \
    .show(truncate=False)


# ------------------------------------------------------------
# STEP 9: VERIFY UNIQUE IDS
# ------------------------------------------------------------

target_final = spark.table("target_customers")

print("TOTAL RECORDS:", target_final.count())

print(
    "DISTINCT RECORD IDS:",
    target_final.select("record_id").distinct().count()
)

print(
    "DISTINCT CUSTOMER NAMES:",
    target_final.select("customer_name").distinct().count()
)


# ------------------------------------------------------------
# STEP 10: CHECK DUPLICATE CUSTOMER NAMES
# ------------------------------------------------------------

target_final.groupBy("customer_name") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


# ------------------------------------------------------------
# STEP 11: UPDATE WATERMARK AFTER SUCCESSFUL MERGE
# ------------------------------------------------------------

new_watermark = incremental_df.agg(
    F.max("updated_at").alias("watermark")
).first()["watermark"]

if new_watermark is not None:
    last_watermark = new_watermark

print("NEW WATERMARK:", last_watermark)

# ============================================================
# END
# ============================================================


INITIAL SOURCE
+-------------+------+---------+-------------------+
|customer_name|amount|status   |updated_at         |
+-------------+------+---------+-------------------+
|Rahul        |1000  |Pending  |2026-10-20 10:00:00|
|Priya        |2000  |Pending  |2026-10-20 11:00:00|
|Amit         |3000  |Shipped  |2026-10-20 12:00:00|
|Sneha        |4000  |Delivered|2026-10-20 13:00:00|
|Kiran        |5000  |Pending  |2026-10-20 14:00:00|
+-------------+------+---------+-------------------+

INITIAL TARGET WITH GENERATED UNIQUE IDS
+------------------------------------+-------------+------+---------+-------------------+
|record_id                           |customer_name|amount|status   |updated_at         |
+------------------------------------+-------------+------+---------+-------------------+
|6b755671-d185-4fd5-8b4f-6306d29e88d3|Rahul        |1000  |Pending  |2026-10-20 10:00:00|
|91e6a9be-c90f-4c08-b170-cdc14087ac2e|Priya        |2000  |Pending  |2026-10-20 11:00:00|
|464b1c2a-7a40-4

In [0]:
b1c3a50e-7d4f-4d2a-8f63-42d1c52e4b90.

In [0]:
Scenario	Matching customer_name	record_id	Action
Existing customer	Priya	Existing UUID	Update
Existing customer	Kiran	Existing UUID	Update
New customer	Vijay	New UUID	Insert
New customer	Anjali	New UUID	Insert